# 🔀 混合推理优化 — 各框架如何应对工具调用与多模态

**前置阅读**：建议先读完 `03-gpu-memory-layout.ipynb`（理解多模态和工具调用的显存冲击）和 `05-transformer-architecture.ipynb`（理解架构基础）。

**本文目标**：深入对比 vLLM、SGLang、TensorRT-LLM、llama.cpp 四大框架在**混合推理**场景下的优化设计——当一个服务同时处理纯文本、多轮工具调用、图片输入时，每个框架如何做显存管理、前缀缓存和调度。

读完这篇你会理解：

- 混合推理为什么比纯文本推理难一个数量级
- vLLM 的 Automatic Prefix Caching (APC) 在工具调用中的表现
- SGLang 的 RadixAttention 为什么是 prefix sharing 的 SOTA
- TensorRT-LLM 的 in-flight batching 如何在异构请求中保持高吞吐
- llama.cpp 的静态分配策略在混合场景下的取舍
- 生产环境中如何选型和配置

## 1. 混合推理为什么难？

### 1.1 三种请求类型，三种资源需求

一个典型的 LLM 推理服务可能同时处理：

```
请求类型 A: 纯文本对话
  "写一首关于春天的诗"
  → Prefill: 5 tokens, Decode: ~100 tokens
  → 轻量级，KV Cache 增长慢
  → Memory-bound (Decode)

请求类型 B: 工具调用 (Agent)
  函数定义: ~2,000 tokens (system prompt)
  多轮交互: 每轮 user + tool_call + tool_result + assistant
  → Prefill: 可能 5,000-15,000 tokens (取决于 tool_result 大小)
  → KV Cache: 随轮数线性累积，可能超 10,000 tokens
  → 混合 compute-bound (prefill) 和 memory-bound (decode)

请求类型 C: 多模态 (图片理解)
  图片: 1-10 张 → 576-5,760 visual tokens
  文本: 简短指令 ~20 tokens
  → Prefill: 处理大量 visual tokens → 重度 compute-bound
  → KV Cache: visual tokens 占大头，但 decode 短
  → Visual encoder 额外 ~2GB 显存
```

### 1.2 三大矛盾

```
矛盾 1: Prefill 和 Decode 的资源冲突
  ┌─────────────────────────────────────────────────────┐
  │ 请求 B 正在进行长 prefill (工具结果注入, 5K tokens)    │
  │ 请求 A 第 87 步 decode → 只生成 1 个 token            │
  │                                                       │
  │ GPU SM:   想给 prefill 满算力 (compute-bound)          │
  │           但又不能阻塞 decode 请求 (延迟 SLA)           │
  │ 问题:     如何在同一 GPU 上同时高效处理两者?            │
  └─────────────────────────────────────────────────────┘

矛盾 2: 显存分配的可预测性
  ┌─────────────────────────────────────────────────────┐
  │ 10 个纯文本请求 4K: KV Cache ~20 GB → 可预测           │
  │ 5 个 Agent + 3 个多模态 + 2 个纯文本:                  │
  │   Agent 可能 5K-50K tokens (取决于工具调用轮数)         │
  │   多模态 可能 576-5,760 visual tokens (取决于图片数)    │
  │   → 显存使用完全不可预测!                              │
  └─────────────────────────────────────────────────────┘

矛盾 3: 前缀复用的粒度
  ┌─────────────────────────────────────────────────────┐
  │ 所有 Agent 请求共享 system prompt (函数定义)           │
  │   → prefix caching 显然应该做                        │
  │ 但 tool_result 各不相同 → 后续部分无法共享            │
  │ 同一张图片被多个请求引用 → visual tokens 应该共享       │
  │   → 但不同请求的 text prompt 不同 → 只能共享前半段     │
  └─────────────────────────────────────────────────────┘
```

### 1.3 各框架的应对思路一览

| 框架 | 核心武器 | 最适合场景 |
|------|---------|----------|
| **vLLM** | PagedAttention + APC | 通用 API 服务，多模型 |
| **SGLang** | RadixAttention + 结构化生成 | Agent 密集、前缀复用多 |
| **TensorRT-LLM** | In-flight batching + 编译优化 | 固定模型、极致吞吐 |
| **llama.cpp** | 静态分配 + KV Cache 量化 | 消费级硬件、本地推理 |

## 2. vLLM — PagedAttention + APC 的混合推理方案

### 2.1 PagedAttention 在混合场景下的优势

回顾 PagedAttention 的核心思想：将 KV Cache 分成固定大小的 **block**（类似 OS 的 page），通过 **block table** 映射逻辑位置到物理位置。

```
纯文本请求 (4K tokens):
  Block table: [b0, b1, b2, b3, b4, b5, b6, b7]  (8 blocks × 512 tokens)
  → 简单的线性映射

Agent 请求 (动态增长, 最终 12K tokens):
  Block table: [b0, b1, ..., b23]  (24 blocks)
  → block 按需分配, 用完回收, 无内部碎片

多模态请求 (5 张图, 2,880 visual tokens + 100 text tokens):
  Block table: [b0-b5 (visual), b6 (text), b7-b9 (decode)]
  → visual tokens 和 text tokens 在同一个 block 空间管理
```

**关键优势**：PagedAttention 的 block 抽象让**不同类型 token 的 KV Cache 可以在同一个显存池中统一管理**——visual tokens、text tokens、函数定义、tool results 都是 block，分配回收逻辑完全一致。

### 2.2 Automatic Prefix Caching (APC)

vLLM 的 APC 是处理工具调用的核心优化。

**工作原理**：

```
场景: 10 个 Agent 请求, 共享同一套 function definitions (2,000 tokens)

无 APC:
  每个请求独立存储 system prompt 的 KV Cache
  → 10 × 2,000 tokens = 20,000 tokens 的 KV Cache (~10 MB)
  → 而且每个请求都需要重新 prefill system prompt

有 APC:
  对 KV Cache block 做内容寻址 (content-based hashing)
  ┌──────────────────────────────────────────────┐
  │ System Prompt: "You have the following tools:│
  │   - get_weather(city)                        │
  │   - search_web(query)                        │  ← 2,000 tokens
  │   - calculate(expression)                    │
  │   ..."                                       │
  └──────────────────────────────────────────────┘
        │
        ▼  hash(token sequence of each block)
  Block 0: hash_abc123 → [已缓存? 否 → 计算并存储]
  Block 1: hash_def456 → [已缓存? 否 → 计算并存储]
  Block 2: hash_ghi789 → [已缓存? 否 → 计算并存储]
  Block 3: hash_jkl012 → [已缓存? 否 → 计算并存储]

  请求 2 到达, 同样的 system prompt:
  Block 0: hash_abc123 → [已缓存! → 直接复用, 跳过计算 ✓]
  Block 1: hash_def456 → [已缓存! → 直接复用, 跳过计算 ✓]
  ...

  效果:
  - 10 个请求的 system prompt KV Cache: 只存 1 份! (2,000 tokens)
  - 每个新请求的 system prompt prefill: 跳过! (命中缓存)
  → 显存节省 90%, prefill 时间节省 90%
```

**APC 的局限性**（在工具调用场景中）：

```
请求 1: [System Prompt (2K)] [User: "北京天气?"] [Tool: get_weather]
        [Result: 北京35度] [Assistant: "北京35度..."]
          ↑ cache hit    ↑ cache hit  ↑ cache miss (tool_result 内容不同)

请求 2: [System Prompt (2K)] [User: "上海天气?"] [Tool: get_weather]
        [Result: 上海32度] [Assistant: "上海32度..."]
          ↑ cache hit    ↑ cache miss ↑ cache miss  ← 用户问题不同

APC 的有效性取决于前缀重合度:
  System prompt (函数定义): 几乎 100% 命中 ✓
  用户问题: 几乎 0% 命中 ✗
  Tool result: 几乎 0% 命中 ✗（结果每次都不同）
  → 实际命中率: ~2,000 / (2,000 + user + result + assistant) ≈ 20-40%
  → 有收益，但没有想象的那么大
```

### 2.3 vLLM 的多模态支持

vLLM 通过 `MultiModalRegistry` 将 visual encoder 的结果注入到 LLM 的 prefill 阶段：

```
流程:
  1. 用户上传图片
  2. vLLM 调用 visual encoder (CLIP/ViT) → visual embeddings
  3. visual embeddings 经过 projector → LLM 维度
  4. 拼接到 text embeddings 前面 → 送入 LLM
  5. LLM 的 prefill 阶段一次性处理 visual + text tokens
  6. visual tokens 的 KV Cache 进入 PagedAttention 管理的 block 池

显存特点:
  - Visual encoder 权重 (~2GB) 常驻显存
  - Visual feature map (prefill 阶段临时): ~0.5-2 GB
  - Visual token KV Cache: 与 text tokens 统一管理
  - 如果多请求引用同一张图 → 需要手动实现 visual token 复用
    (vLLM 当前版本对此支持有限，这是 vs SGLang 的差距之一)
```

### 2.4 vLLM 的调度策略

vLLM scheduler 对混合请求的处理：

```
请求到达 → 进入 waiting queue

调度逻辑 (每次 forward 之前运行):
  1. 先尝试从 waiting queue 拉请求做 prefill
     - 检查显存: 是否有足够的 free blocks?
     - 检查 slot: 当前 batch 的 token budget?
     - 优先级: 先到先服务 (FCFS)

  2. 分配 KV Cache blocks
     - 纯文本: 预估 max_tokens 个 block
     - Agent: 同上 (难以预估, 可能导致 block 浪费)
     - 多模态: visual token blocks + 预估 text blocks

  3. Running 请求的 decode 照常进行
     - 如果需要抢占 (OOM 风险): swap 被抢占请求的 KV Cache → CPU

混合场景的问题:
  - FCFS 调度: 长 prefill 的多模态请求会阻塞短请求
  - vLLM v0.6.0+ 引入了 priority-based scheduling
  - 但仍不如 TensorRT-LLM 的 in-flight batching 灵活
```

## 3. SGLang — RadixAttention 与结构化生成

SGLang 是目前在**前缀复用**维度上做得最极致的框架。

### 3.1 RadixAttention：基于 Radix Tree 的前缀缓存

与 vLLM 的 APC（块级 hash 匹配）不同，SGLang 使用 **radix tree**（压缩前缀树）来管理所有请求的 KV Cache：

```
                     ┌─────────────────────┐
                     │  Root (空)            │
                     └──────┬──────────────┘
                            │
              ┌─────────────┴──────────────┐
              │  System Prompt: "You are   │
              │  a helpful assistant..."   │  ← 所有请求共享这个节点
              │  (1,500 tokens)            │
              └──────┬─────────────────────┘
                     │
         ┌───────────┴───────────┐
         │                       │
  ┌──────┴──────┐         ┌──────┴──────┐
  │ "北京天气?"  │         │ "分析这张图" │
  │ (user msg)   │         │ (user msg)   │
  └──────┬──────┘         └──────┬──────┘
         │                       │
    ┌────┴────┐            ┌─────┴─────┐
    │tool_call│            │ image_emb │
    │get_     │            │ (576 vis  │
    │weather  │            │  tokens)  │
    └────┬────┘            └─────┬─────┘
         │                       │
    ┌────┴────┐            ┌─────┴─────┐
    │result:  │            │assistant: │
    │北京35度  │            │"这张图... │
    └─────────┘            └───────────┘

Radix Tree 的特性:
  1. 自动发现公共前缀: 不需要手动指定哪些 token 应该共享
  2. 部分匹配: 即使两个请求不完全相同, 共享前缀自动复用
  3. 动态剪枝: 当节点的引用计数为 0 时, 回收其 KV Cache
  4. O(log n) 查找: 找到最长的公共前缀
```

### 3.2 RadixAttention vs vLLM APC 的关键差异

**场景: 5 个 Agent 请求, 函数定义相同但用户问题不同**

```
vLLM APC (block-level hash):
  [System Prompt Block 0] [Block 1] [Block 2] [Block 3]
        ↑ hash match      ↑ match    ↑ match    ↑ match
  → System Prompt 完美命中 ✓
  
  [User Q1 Block 4]  [Tool Block 5]  [Result Block 6]  [Asst Block 7]
        ↑ miss            ↑ miss         ↑ miss            ↑ miss
  → 后面全部 miss ✗

SGLang RadixAttention (token-level radix tree):
  System Prompt 作为一个 radix tree 子树 → 所有请求共享 ✓
  在 System Prompt 之后, radix tree 分叉:
    Path A: User Q1 → Tool → Result A → Asst A
    Path B: User Q2 → Tool → Result B → Asst B
    ...
  → 如果 User Q1 和 User Q2 有共同前缀 (如 "查询今天"), 也能部分复用
  
关键差异:
  APC:  block 边界切割, 必须从第一个 token 开始才能命中
        如果只有中间的 block 相同 → 无法复用
  
  Radix: token 级别匹配, 任意长度的公共前缀都能被利用
          一个请求可以"骑"在另一个请求的树的中间节点上
```

### 3.3 工具调用的结构化生成

SGLang 的另一个杀手锏是 **constrained decoding**——让 LLM 严格按照 JSON Schema 生成工具调用：

```
# 普通 generation (可能出错):
assistant: "Let me call get_weather with city='Beijing'"  ← 需要解析, 容易格式错误

# SGLang constrained generation (保证正确):
定义: {"type": "function", "function": {"name": "get_weather", 
        "parameters": {"city": {"type": "string"}}}}
        
生成时 SGLang 构建 token 级别的 FSM (有限状态机):
  State 0: 必须输出 '{"function": {"name": "'
  State 1: 从 [tool names] 中选择 → "get_weather"
  State 2: 必须输出 '", "parameters": {"city": "'
  State 3: 自由文本 → "Beijing"
  State 4: 必须输出 '"}}'

效果:
  - 100% 合法的 JSON 输出, 不需要 retry
  - 跳过了不符合 Schema 的 token → 实际减少了无效 decode
  - 在 Agent 密集型场景 (每轮都有 tool_call) 中收益巨大
```

这在推理优化角度的含义：
- 跳过的 token (不符合 Schema 的) → 不需要算 → **实际加速**
- 不需要 retry → 减少了浪费的 decode steps → **节省 KV Cache**
- 更重要的是：**约束生成 + RadixAttention 是一对天然组合**——函数定义不仅是 prefix cache 的热点，constrained generation 还保证了 tool_call 的格式一致性，进一步提高了不同请求间 token 序列的相似度 → 更高的 cache 命中率

### 3.4 SGLang 的多模态支持

```
SGLang 多模态管线:

1. 图像预处理 (CPU/GPU)
   image → ViT/CLIP → visual embeddings

2. Visual token 注入 (类似 vLLM)
   [visual_embeds | text_embeds] → LLM

3. RadixAttention 自动管理 visual token KV Cache
   - 多请求引用同一张图 → visual tokens 共享! ← 比 vLLM 强
   - Radix tree 自动检测: 如果两个请求的 visual embeddings 相同
     → 它们的 KV Cache 被合并到 radix tree 的同一节点
   - 例如: 10 个用户上传了同一张 viral image
     → visual token KV Cache 只存 1 份!

4. 图片和文本的混合前缀缓存
   [System Prompt] → [Image Embedding] → [User Text] → ...
   如果 3 个请求的 [System + Image] 相同, radix tree 自动共享
```

**SGLang 针对多模态的额外优化**：

```
RadixAttention 的 image-aware 特性:
  同一张图的 visual tokens 在不同压缩比下,
  radix tree 能处理"部分匹配":
  
  请求 A: 图片全分辨率 → 576 visual tokens
  请求 B: 同一张图, 降采样 → 256 visual tokens
  → 虽然 visual tokens 不完全相同, 但 radix tree
    可以共享 text 部分 (system prompt 等)
```

## 4. TensorRT-LLM — In-flight Batching 与编译优化

### 4.1 In-flight Batching 解决混合请求的调度问题

这是 TensorRT-LLM 最独特的武器。与 vLLM 的 continuous batching（每步开关 batch）不同，in-flight batching 允许 **prefill 和 decode 在同一个 forward pass 中混合处理**：

```
vLLM Continuous Batching:
  Step 0: [prefill: req0(5K)]                          ← 这一步只做 prefill
  Step 1: [decode: req0(1)] [prefill: req1(2K)]        ← prefill + decode 同批
  Step 2: [decode: req0(1)] [decode: req1(1)]           ← 全部 decode

  prefill 时整个 batch 都在等 → decode 延迟被拖高

TensorRT-LLM In-flight Batching:
  Step 0: [prefill: req0(5K)] [decode: req3(1)] [decode: req4(1)]
          ↑ 长 prefill    ↑ 短 decode 不阻塞!
          
  实现方式:
  - prefill 请求被拆成 chunk (如每 chunk 512 tokens)
  - 每个 chunk 之间可以插入其他请求的 decode
  - 像"时间片轮转"一样交替处理

  效果:
  请求 A (10K prefill):  [chunk0][chunk1]...[chunk19]
  请求 B (decode):       [d][d][d][d][d][d][d]...     ← 不受影响!
  请求 C (工具调用 2K prefill): [chunk0][chunk1][chunk2][chunk3]

  时间线:
  t=0: A预0  B解  C预0
  t=1: A预1  B解  C预1
  t=2: A预2  B解  C预2  ← B 的 decode 从不被阻塞
  ...
```

**这对混合推理意味着什么？**

| 场景 | vLLM (continuous) | TensorRT-LLM (in-flight) |
|------|-------------------|--------------------------|
| 大图 prefill (5K tokens) + 10 个 decode 请求 | decode 请求等 prefill 完成 → P99 延迟飙高 | decode 请求在 chunk 间隙插入 → P99 平稳 |
| Agent 多轮 (频繁 prefill tool result) | 每次 tool result 注入都是一个 prefill 阻塞点 | tool result 注入被 chunked, 不阻塞 |
| 混合负载 (50% text + 30% agent + 20% vis) | 长 prefill 随机阻塞短请求 | 所有请求的延迟可预测 |

### 4.2 TensorRT-LLM 的 KV Cache 管理

```
TensorRT-LLM 的 KV Cache 管理 = 类似 PagedAttention + 编译时优化

Block 管理:
  - 类似 vLLM 的 block-based 分配: 固定大小 blocks, block table
  - 额外优化: 编译时确定 block 大小最优值
    (vLLM 的 block size 是固定的 16, TensorRT-LLM 可自动调优)

Prefix Caching:
  - TensorRT-LLM 也支持 prefix caching
  - 但采用的是 exact-match approach (类似 APC)
  - 不如 SGLang 的 RadixAttention 粒度细
  - 编译时可以为已知的 system prompt 预分配 + 预计算 KV Cache

Memory Pool:
  - 启动时构建 CUDA graph → 预分配固定大小的 memory pool
  - 优点: 零运行时内存碎片
  - 缺点: 灵活性受限, 多模态/工具调用的不可预测性可能导致 pool 不够用
```

### 4.3 编译优化在混合场景下的额外收益

因为 TensorRT-LLM 对模型图做编译时优化，它能做 kernel fusion 来加速一些混合推理特有的模式：

```
工具调用场景的 kernel fusion 机会:

  频繁出现的 pattern: [user text] → [tool_call JSON] → [EOS? or continue]
  
  TensorRT-LLM 可以发现并优化:
  - Embedding lookup + LayerNorm 融合
  - Attention + Residual + LayerNorm 融合 (已经在做)
  - Gated MLP (SwiGLU) 的 kernel fusion
  
  多模态场景:
  - Visual encoder 的卷积操作 → TensorRT 编译加速
  - Cross-modal projection → 融合成一个 kernel

但代价:
  - 每次换模型都要重新编译 → 不适合频繁切换模型
  - 编译时优化依赖图结构不变 → 动态 batch/dynamic shape 需要额外处理
```

### 4.4 TensorRT-LLM 的多模态支持 (Multimodal Plugin)

```
架构:
  ┌─────────────────────────────┐
  │  TensorRT-LLM Runtime        │
  │  ┌───────────────────────┐   │
  │  │  LLM Engine (TRT)      │   │
  │  │  • Attention ops       │   │
  │  │  • MLP ops             │   │
  │  │  • KV Cache manager    │   │
  │  └───────────────────────┘   │
  │            ↕                  │
  │  ┌───────────────────────┐   │
  │  │  Multimodal Plugin     │   │  ← 独立编译的视觉组件
  │  │  • ViT/CLIP (TRT opt)  │   │
  │  │  • Projector           │   │
  │  │  • Visual tokenizer    │   │
  │  └───────────────────────┘   │
  └─────────────────────────────┘

优点: LLM 和 visual encoder 都经过 TensorRT 编译 → 端到端高性能
缺点: 多模态模型切换需要重新编译 visual plugin → 不够灵活
```

## 5. llama.cpp — 静态分配与极致量化

llama.cpp 的设计哲学与其他三个框架完全不同：**面向消费级硬件，内存优先，简单可靠**。

### 5.1 静态 KV Cache 分配

```
llama.cpp 的显存模型:

启动时 (model load):
  ┌──────────────────────────────────────┐
  │ 模型权重 (GGUF)                       │
  │ (Q4_K_M 7B: ~4 GB)                   │
  ├──────────────────────────────────────┤
  │ KV Cache (预分配, 固定大小)            │
  │ 计算: n_layers × max_seq_len ×       │
  │        n_kv_heads × head_dim × dtype │
  │                                       │
  │ 例如: 32 × 8192 × 32 × 128 × 1B      │
  │       (Q8_0 KV Cache)                │
  │     ≈ 1 GB (per slot)                │
  │                                       │
  │ 如果 -c 8192 -np 4:                   │
  │   4 slots × 1 GB = 4 GB              │
  ├──────────────────────────────────────┤
  │ 激活值 + 临时 buffer                   │
  │ ~0.5-1 GB                            │
  ├──────────────────────────────────────┤
  │ Visual Encoder (llava 多模态)          │
  │ ~2 GB (可选, 用完可释放)               │
  └──────────────────────────────────────┘

对比:
  vLLM:  dynamic allocation (blocks)
  llama:  static pre-allocation (fixed buffer per slot)
```

### 5.2 静态分配在混合场景下的取舍

```
缺点 (明显):
  - 只有 N 个 slot → 最多 N 个并发请求
  - slot 用满 → 拒绝新请求 (不像 vLLM 可以动态扩展)
  - 短请求也占用整个 slot → 内部浪费
  - 工具调用的长 context 可能超出 max_seq_len → 截断

优点 (被低估):
  - 0 碎片 → 显存利用可预测, 不会 OOM
  - 0 运行时分配开销 → decode 延迟更稳定
  - 简单 = 更少的 bug surface
  - KV Cache Q8_0 量化 → 同样的 slot size 可以支持更长的 context

多模态 (llava):
  llama.cpp 的 llava 实现在处理图片时:
  1. 加载 visual encoder (临时)
  2. 处理图片 → visual embeddings
  3. 拼接到 text embeddings 前面
  4. 释放 visual encoder (如果显存紧张)
  5. LLM 正常处理 (visual tokens 进入 KV Cache)
  
  注意: visual tokens 占用了 slot 的 KV Cache 空间!
  如果 -c 4096, 5 张图 × 576 = 2,880 visual tokens
  → 只剩 4096 - 2,880 = 1,216 tokens 给文本和生成的 token
```

### 5.3 llama.cpp 的工具调用实践

```
llama.cpp 本身不内置 function calling, 但可以通过:

1. llama.cpp server (OpenAI-compatible API):
   - --jinja 或 --chat-template 支持 tool calling
   - 在 application 层解析 tool_call → 执行 → 注入 result
   - KV Cache 在整个会话中保留 (slot 一直占用)

2. 会话管理策略:
   简单粗暴: 1 会话 = 1 slot
   → 用户的整个多轮工具调用在同一个 slot 中
   → slot 的 max_seq_len 是所有 token 的上限
   
   如果 -c 8192:
     系统提示 + 函数定义:   1,500 tokens
     Round 1:              500 tokens (user + tool_call + result + assistant)
     Round 2-5:           2,000 tokens
     累计:                4,000 tokens → 还在安全范围内
     
     Round 6-10:          +2,500 tokens
     累计:                6,500 tokens → 接近上限
     
   → 对 Agent 场景, max_seq_len 是硬限制
   → 需要仔细管理 context: 截断/摘要/滑动窗口

3. llama.cpp 的特殊优势:
   - 可以在 MacBook 上跑 → 本地 Agent 开发
   - 极低的硬件成本 → 适合个人项目
   - KV Cache Q8_0 量化 → 8K context 可以在 8GB 显存上实现
```

## 6. 四大框架横向对比

### 6.1 核心机制对比

| 维度 | vLLM | SGLang | TensorRT-LLM | llama.cpp |
|------|------|--------|-------------|-----------|
| KV Cache 管理 | PagedAttention (blocks) | RadixAttention (radix tree) | Block-based + compile-time | **Static buffer (fixed slots)** |
| 前缀缓存 | APC (block hash) | **Radix tree (token-level)** | Exact match | 无 (每个 slot 独立) |
| 调度策略 | Continuous batching | Continuous batching | **In-flight batching** | Slot-based FCFS |
| 请求混合 | Prefill + Decode 同批次 | 同 vLLM | **Prefill chunked, 插缝 Decode** | 序列化处理 |
| 显存模型 | Dynamic (blocks 按需) | Dynamic (radix tree nodes) | Semi-static (pre-alloc + graph) | **Fully static** |
| 多模态 | MultiModalRegistry | RadixAttention 共享 visual KV | Multimodal Plugin (TRT compiled) | llava (临时 encoder) |
| 量化 | AWQ/GPTQ/FP8 (后端) | 同 vLLM | **原生 FP8/INT8/INT4** | **GGUF (K-quant 系列)** |

### 6.2 场景匹配度

```
场景: 高并发纯文本 API (如 ChatGPT-like 服务)
  ┌──────┬──────┬─────────────┬──────────┐
  │ vLLM │SGLang│ TensorRT-LLM│ llama.cpp│
  │ ★★★★★│ ★★★★ │ ★★★★★      │ ★★      │
  └──────┴──────┴─────────────┴──────────┘
  推荐: vLLM (生态最好) 或 TensorRT-LLM (极致吞吐)

场景: Agent 密集型 (多轮工具调用, 前缀复用多)
  ┌──────┬──────┬─────────────┬──────────┐
  │ vLLM │SGLang│ TensorRT-LLM│ llama.cpp│
  │ ★★★  │ ★★★★★│ ★★★★       │ ★★★     │
  └──────┴──────┴─────────────┴──────────┘
  推荐: SGLang (RadixAttention + constrained decoding 是绝配)

场景: 多模态服务 (图片+文本混合)
  ┌──────┬──────┬─────────────┬──────────┐
  │ vLLM │SGLang│ TensorRT-LLM│ llama.cpp│
  │ ★★★★ │ ★★★★★│ ★★★★       │ ★★★     │
  └──────┴──────┴─────────────┴──────────┘
  推荐: SGLang (visual token KV Cache 共享) 或 vLLM (生态)

场景: 混合负载 (不可预测的请求类型)
  ┌──────┬──────┬─────────────┬──────────┐
  │ vLLM │SGLang│ TensorRT-LLM│ llama.cpp│
  │ ★★★  │ ★★★★ │ ★★★★★      │ ★★      │
  └──────┴──────┴─────────────┴──────────┘
  推荐: TensorRT-LLM (in-flight batching 最擅长处理异构请求)

场景: 本地开发 / 消费级 GPU / 隐私优先
  ┌──────┬──────┬─────────────┬──────────┐
  │ vLLM │SGLang│ TensorRT-LLM│ llama.cpp│
  │ ★★   │ ★    │ ★           │ ★★★★★   │
  └──────┴──────┴─────────────┴──────────┘
  推荐: llama.cpp / Ollama (唯一能在 8GB 卡上跑的选择)
```

### 6.3 前缀缓存在不同场景下的命中率估算

下面用前面的模型来估算各框架在混合场景下的缓存效率：

```
场景: 100 个 Agent 请求, 3 个工具, 5 轮交互
  每请求 token 分布:
    System Prompt (函数定义):     2,000 tokens  (所有请求 100% 相同!)
    User message:                  20 tokens   (各不相同)
    Tool call (assistant):         30 tokens   (高度相似, 但格式固定)
    Tool result:                1,000 tokens   (各不相同)
    Assistant response:           100 tokens   (各不相同)
    ... 重复 5 轮 ...
    总计: ~10,000 tokens per request

各框架的缓存命中:
  vLLM APC:
    命中: System Prompt (2,000) × 100 = 200,000 tokens → 只存 2,000 ✓
    未命中: 其余 8,000 tokens × 100 请求 = 800,000 tokens ✗
    命中率: 200K / 1M = 20%
    实际 KV Cache: 2,000 + 800,000 = 802,000 tokens

  SGLang RadixAttention:
    命中: System Prompt (2,000) ← 共享 ✓
    命中: Tool call (30 × 5 = 150) ← 格式固定, 大部分共享 ✓
    假设 80% 的 tool_call token 可共享: 150 × 0.8 × 99 = 11,880 tokens saved
    实际 KV Cache: 802,000 - 11,880 ≈ 790,000 tokens
    命中率: ~22% ← 只比 APC 好一点 (工具调用场景下差距不大)

  TensorRT-LLM:
    命中: 同 APC level (~20%)
    差异: 胜在调度延迟, 不是缓存数量

关键洞察:
  在工具调用场景下, 前缀缓存的收益被高估了!
  System prompt 确实是缓存的 sweet spot, 但它只占总 token 的 20%
  tool_result 每次不同 → 无法缓存 → 这是 KV Cache 的大头

  真正节省显存的是:
  1. System prompt 的 KV Cache 共享 (所有框架都在做)
  2. KV Cache 量化 (llama.cpp/TensorRT-LLM)
  3. 动态回收完成的请求 (所有框架)
```

## 7. 各框架的显存布局对比

### 7.1 同一场景的显存剖面

假设场景：10 并发请求 (4 个纯文本 2K, 3 个 Agent 8K, 3 个多模态 5 图)，LLaMA-7B AWQ INT4，A100-80GB

```
vLLM:
┌──────────┬────────────────────────────────┬──────┬──────┐
│ Weights  │         KV Cache Blocks         │ Act  │ Misc │
│ ~4 GB    │  ~55 GB                          │ ~3GB │ ~3GB │
│          │ [text blocks] [agent blocks]     │      │      │
│          │ [visual blocks] [free blocks]    │      │      │
│          │ ← PagedAttention 统一管理 →      │      │      │
└──────────┴────────────────────────────────┴──────┴──────┘
  利用率: ~65/80 GB (81%)
  碎片率: 低 (block size 16 → 最小碎片 16 tokens)

SGLang:
┌──────────┬────────────────────────────────┬──────┬──────┐
│ Weights  │       Radix Tree KV Cache       │ Act  │ Misc │
│ ~4 GB    │  ~55 GB                          │ ~3GB │ ~3GB │
│          │ [System Prompt subtree]          │      │      │
│          │ [visual token nodes (shared!)]   │      │      │
│          │ [per-request text nodes]         │      │      │
│          │ ← radix tree 自动合并共享节点 →  │      │      │
└──────────┴────────────────────────────────┴──────┴──────┘
  利用率: ~60/80 GB (75%)  ← 比 vLLM 少因为 visual tokens 共享
  碎片率: 极低 (token 级别管理)

TensorRT-LLM:
┌──────────┬──────────────────┬────────────────┬──────┬──────┐
│ Weights  │  KV Cache Pool   │  CUDA Graphs   │ Act  │ Misc │
│ ~4 GB    │  ~50 GB           │  ~5 GB          │ ~3GB │ ~3GB │
│          │ (预分配, 固定大小) │ (编译时图缓存)   │      │      │
└──────────┴──────────────────┴────────────────┴──────┴──────┘
  利用率: ~65/80 GB (81%)
  碎片率: 0 (预分配) 但灵活性最低

llama.cpp:
┌──────────┬──────────────────────────┬──────┬──────┐
│ Weights  │  KV Cache Slots           │ Act  │ Misc │
│ Q4_K_M   │  10 slots × 8K max_len   │ ~1GB │ ~1GB │
│ ~4 GB    │  ~45 GB (Q8_0 KV Cache)   │      │      │
│          │  [slot0] [slot1] ...      │      │      │
│          │  ← 每个 slot 固定大小 →   │      │      │
└──────────┴──────────────────────────┴──────┴──────┘
  利用率: ~51/80 GB (64%) ← 短请求浪费 slot 内的未用 KV Cache
  碎片率: 高 (内部碎片: 短请求只用了 20% 的 slot)
  但: 如果全部是 8K 请求 → 利用率接近 100%
```

### 7.2 混合场景下的显存压力测试

```
各框架在不同混合比例下的表现:

场景 A: 100% 短文本 (2K tokens, 50 并发)
  vLLM:        ✅ 高效 (block 粒度, 几乎 0 浪费)
  SGLang:      ✅ 高效 (同 vLLM)
  TRT-LLM:     ✅ 高效
  llama.cpp:   ⚠️  浪费 (每个 slot 按 max_len 分配)

场景 B: 30% Agent (8K) + 70% 短文本 (2K), 30 并发
  vLLM:        ✅ 动态适应
  SGLang:      ✅ 动态适应 + prefix 共享
  TRT-LLM:     ✅ in-flight batching 让短请求不受长 Agent 影响
  llama.cpp:   ❌ 需要按最长的请求配置 slots, 短请求浪费很多

场景 C: 50% 多模态 (5 图) + 50% 纯文本, 20 并发
  vLLM:        ⚠️  visual tokens 不共享, 重复存储
  SGLang:      ✅ visual tokens 可共享 (如果同图)
  TRT-LLM:     ⚠️  同 vLLM, 但调度更好
  llama.cpp:   ❌ visual tokens 消耗 slot 的 max_len 配额

场景 D: Agent + 多模态混合
  vLLM:        ⚠️  调度不可预测, P99 延迟高
  SGLang:      ✅ RadixAttention + constrained decoding 的组合优势
  TRT-LLM:     ✅ in-flight batching 延迟最稳定
  llama.cpp:   ❌ 只能用于轻量场景
```

## 8. 生产环境选型决策树

```
你的硬件和场景是？

NVIDIA GPU (A100/H100), 多模型, 灵活部署?
├─ 主要是纯文本 → vLLM (默认选择, 生态最完善)
├─ 大量 Agent / 工具调用 → SGLang (RadixAttention + constrained decoding)
├─ 多模态为主 → SGLang (visual token 共享) 或 vLLM
├─ 混合负载, 需要低延迟 SLA → TensorRT-LLM (in-flight batching)
└─ 预算有限, 追求性价比 → vLLM + AWQ INT4

NVIDIA GPU (消费级: RTX 3090/4090)?
├─ llama.cpp + CUDA backend + Q4_K_M
├─ Ollama (包装了 llama.cpp, 易用性更好)
└─ 模型 ≤ 7B, 可用 vLLM (但需要 10GB+ 显存)

Apple Silicon / CPU only?
└─ llama.cpp (唯一选择, 但做得非常好)

### 关键配置建议

针对工具调用场景:
  vLLM:
    --enable-prefix-caching           # 开启 APC
    --max-model-len 16384            # 设置合理上限
    --gpu-memory-utilization 0.90    # 留 10% 给突发

  SGLang:
    --enable-radix-cache             # 开启 RadixAttention
    --schedule-policy priority        # 优先级调度 (Agent 优先)
    --constrained-json-whitespace-pattern  # 优化 JSON 生成

  TensorRT-LLM:
    --max-input-len 16384
    --max-batch-size 32              # 根据显存调整
    --kv-cache-free-gpu-mem-fraction 0.95

  llama.cpp server:
    -c 16384                          # 比预期的最大 context 多 20%
    -np 8                             # 并发 slots
    --cache-type-k q8_0 --cache-type-v q8_0  # KV Cache 量化
    --mlock                           # 锁定内存防止 swap

针对多模态场景:
  vLLM / SGLang:
    --limit-mm-per-prompt image=5     # 限制单请求图片数
    # 确保 visual encoder 的高效利用
  
  TensorRT-LLM:
    # 需要预先编译 visual encoder 的 TRT engine
    # 图片尺寸标准化 → 减少 visual token 数量差异

  llama.cpp:
    # 使用 llava 或 minicpm-v 等 GGUF 多模态模型
    # 注意 -c 的值: 要包含 visual tokens 的配额
```

## 9. 代码实验：模拟各框架的 KV Cache 效率

模拟 100 个 Agent 请求在不同框架策略下的 KV Cache 使用量。

In [ ]:
# 各框架在工具调用场景下的 KV Cache 效率模拟

import random
from collections import defaultdict

class CacheSimulator:
    def __init__(self, block_size=16, name="Generic"):
        self.block_size = block_size
        self.name = name
        self.stored_blocks = {}  # hash → block content (for reference)
        self.total_stored = 0
        self.total_requested = 0
        self.hits = 0
        self.misses = 0

    def process_request(self, tokens):
        """处理一个请求, 返回 KV Cache token 数"""
        self.total_requested += len(tokens)
        stored_for_this_request = 0

        for i in range(0, len(tokens), self.block_size):
            block = tuple(tokens[i:i+self.block_size])
            block_hash = hash(block)
            self.hits += 1
            stored_for_this_request += len(block)

            # First time seeing this prefix → store
            if block_hash not in self.stored_blocks:
                self.stored_blocks[block_hash] = block
                self.total_stored += len(block)
                self.hits -= 1
                self.misses += 1

        return stored_for_this_request


class RadixCacheSimulator:
    """简化版 Radix Tree 缓存模拟"""
    def __init__(self, name="RadixAttention"):
        self.name = name
        self.trie = {}  # token → subtrie
        self.total_stored = 0
        self.total_requested = 0

    def process_request(self, tokens):
        self.total_requested += len(tokens)
        node = self.trie
        stored = 0

        for t in tokens:
            if t not in node:
                node[t] = {}
                self.total_stored += 1
            else:
                stored += 1  # 这个 token 已经缓存
            node = node[t]

        return stored  # 返回命中的 token 数


def generate_agent_session(system_tokens=2000, rounds=5,
                           user_len=20, tool_call_len=30,
                           result_len=1000, asst_len=100):
    """生成一个 Agent 会话的所有 token"""
    tokens = list(range(system_tokens))  # System prompt (所有请求共享)

    for r in range(rounds):
        tokens.append(10000 + r * 1000 + random.randint(0, 999))  # user
        for _ in range(user_len - 1):
            tokens.append(random.randint(20000, 29999))

        for _ in range(tool_call_len):
            tokens.append(30000 + random.randint(0, 999))  # tool_call

        for _ in range(result_len):
            tokens.append(40000 + random.randint(0, 9999))  # result

        for _ in range(asst_len):
            tokens.append(50000 + random.randint(0, 999))  # assistant

    return tokens


print("=" * 70)
print("各框架 KV Cache 效率模拟: 100 Agent 请求, 5 轮工具调用")
print("=" * 70)

random.seed(42)

# 生成 100 个会话
sessions = [generate_agent_session() for _ in range(100)]
avg_len = sum(len(s) for s in sessions) / len(sessions)
print(f"\n每请求平均 token 数: {avg_len:,.0f}")
print(f"其中 System Prompt: 2,000 tokens (所有请求共享)")
print(f"Tool result 每轮: 1,000 tokens (各不相同)")
print()

# 模拟 vLLM APC (block-level)
apc = CacheSimulator(block_size=16, name="vLLM APC")
for sess in sessions:
    apc.process_request(sess)

# 模拟 SGLang RadixAttention (token-level)
radix = RadixCacheSimulator()
for sess in sessions:
    radix.process_request(sess)

# 模拟无缓存 (baseline)
total_tokens = sum(len(s) for s in sessions)

# 模拟 llama.cpp (static slots, 每个 slot 独立, 无共享)
# 假设 slot_size = 12000, 10 slots
llama_stored = 10 * 12000  # 每个 slot 预分配, 无论实际用量

print("─" * 70)
print(f"{'框架':<25s} {'KV Cache 总量':>15s} {'命中率':>10s} {'节省 vs 无缓存':>15s}")
print("─" * 70)

print(f"{'无缓存 (baseline)':<25s} {total_tokens:>13,} tokens {'--':>10s} {'--':>15s}")

apc_hit_rate = apc.hits / (apc.hits + apc.misses) * 100
apc_saving = (1 - apc.total_stored / total_tokens) * 100
print(f"{'vLLM APC (block=16)':<25s} {apc.total_stored:>13,} tokens {apc_hit_rate:>9.1f}% {apc_saving:>14.1f}%")

radix_stored = radix.total_stored
radix_saving = (1 - radix_stored / total_tokens) * 100
print(f"{'SGLang RadixAttention':<25s} {radix_stored:>13,} tokens {'--':>10s} {radix_saving:>14.1f}%")

# 计算各部分的缓存贡献
print(f"\n缓存命中分解 (SGLang RadixAttention):")
system_hits = 2000 * 99  # System prompt: 第一个请求存储, 后 99 个命中
tool_call_hits_per_req = 30 * 5 * 0.6  # 约 60% 的 tool_call tokens 可共享
tool_call_hits = int(tool_call_hits_per_req * 99)

print(f"  System Prompt 命中:    {system_hits:>10,} tokens (99个请求复用)")
print(f"  Tool call 模式命中:    {tool_call_hits:>10,} tokens (固定格式)")
print(f"  其他 (user/result/asst): {radix_stored - 2000 - int(30*5*0.6):>10,} tokens (全新存储)")
print(f"  总存储:                {radix_stored:>10,} tokens")

print(f"\n💡 关键洞察:")
print(f"  1. System prompt 缓存贡献最大 (~20% 节省)")
print(f"  2. Tool call 格式相似 → 额外 ~2-3% 节省")
print(f"  3. Tool result 每次不同 → 无法缓存 (占总量 ~50%)")
print(f"  4. APC vs Radix: 差距在 2-5%, 主要优势是粒度而非总量")
print(f"  5. 真正省显存的是: KV Cache 量化 + 动态回收, 而不是前缀缓存!")
print(f"  6. llama.cpp 静态分配: 10 slots × 12K = {llama_stored:,} tokens")
print(f"     但如果实际只用 4K/slot → 浪费 {10*8000:,} tokens (67%!)")

## 10. 混合推理的实战优化策略

### 10.1 通用策略（适用所有框架）

```
1. System Prompt 瘦身
   - 压缩函数定义: 去掉不必要的 description, 用简短的命名
   - 示例: "get_weather(city: str) -> dict" 而不是 200 字的描述
   - 收益: 减少 ~30-50% 的函数定义 token 数

2. Tool Result 截断
   - 设置 tool_result 的 max_tokens (如 2000)
   - 超过则截断 + 加 "...[truncated]"
   - 收益: 防止单个 tool result 耗尽 context

3. 轮数限制 + 滑动窗口
   - 设置 max_rounds (如 10 轮)
   - 超过后丢弃最老的轮次 (保留 system prompt)
   - 或使用摘要: 对旧轮次做摘要, 替换原始 token

4. 图片预处理
   - 限制分辨率: 1024×1024 → 512×512 → visual tokens 减少 4x
   - 限制图片数量: max 5 张/请求
   - 统一尺寸: 所有图片 resize 到相同尺寸 → 稳定显存消耗

5. 分离服务 (推荐)
   - 不要把所有请求类型混在一个服务上
   - 纯文本 API → 一个部署 (高并发, 低 KV Cache)
   - Agent 服务 → 另一个部署 (低并发, 高 KV Cache, prefix caching)
   - 多模态服务 → 第三个部署 (含 visual encoder)
   - 收益: 每种部署可以针对性优化配置
```

### 10.2 框架特定的最佳配置

```
vLLM Agent 服务:
  --enable-prefix-caching
  --max-num-seqs 32                # 限制并发, 给 KV Cache 留空间
  --max-model-len 16384
  --gpu-memory-utilization 0.92
  --enforce-eager                   # 如果 CUDA graph 有兼容问题

SGLang Agent 服务:
  # SGLang 默认配置已经针对 Agent 场景优化
  --schedule-policy priority        # 或 lpm (longest prefix match)
  --context-length 16384
  # 不需要 --enable-radix-cache (默认开启)

TensorRT-LLM 混合负载:
  # 编译时:
  --max-input-len 16384
  --max-batch-size 64
  --max-num-tokens 65536            # 每 batch 最大 token 数

  # 运行时:
  --kv-cache-free-gpu-mem-fraction 0.95
  --enable-chunked-context           # 启用 prefill chunked

llama.cpp Agent 开发:
  ./llama-server \
    -m model-q4_k_m.gguf \
    -c 16384 \
    -np 4 \
    --cache-type-k q8_0 --cache-type-v q8_0 \
    --mlock \
    --chat-template tool_calling    # 使用支持 function calling 的模板
```

### 10.3 监控关键指标

```
混合推理需要监控的维度:

  KV Cache 相关:
    - kv_cache_utilization: 当前 KV Cache 使用率
    - prefix_cache_hit_rate: 前缀缓存命中率
    - tokens_per_request: 每请求的平均/最大 token 数
    - visual_tokens_per_request: 多模态请求的 visual token 数

  延迟相关:
    - TTFT P50/P99: 对 Agent 场景特别重要 (高频 prefill)
    - TPOT P50/P99: decode 阶段的稳定性
    - prefill_time_seconds: 长 prefill (tool result 注入) 的耗时

  吞吐相关:
    - requests_per_second
    - tokens_per_second (total + per-request-type)

  显存相关:
    - gpu_memory_used / gpu_memory_total
    - kv_cache_memory_used (按请求类型细分)
    - num_preemptions: OOM 导致的抢占次数 → 越少越好
```

## 11. 总结

### 核心结论

1. **前缀缓存在工具调用场景的收益被高估了**：system prompt 的共享收益约 20%，但 tool_result 每次不同，无法缓存。真正省显存的是 KV Cache 量化和动态回收。

2. **SGLang 是 Agent 密集型场景的最佳选择**：RadixAttention + constrained decoding 的组合优势，在工具调用场景下没有对手。

3. **TensorRT-LLM 是混合负载（不可预测请求类型）的王者**：in-flight batching 保证所有请求类型的延迟都可预测。

4. **vLLM 是通用默认选择**：生态最完善，除非你有明确的理由选择其他框架。

5. **llama.cpp 在消费级硬件上无可替代**：静态分配虽然浪费，但简单可靠，能在 8GB 显存上跑 7B 模型 + 工具调用。

### 阅读路径

完成本文后：

- 进入 `local/` → llama.cpp deep dive（理解静态 KV Cache 分配的实现细节）
- 进入 `server/` → vLLM deep dive（理解 PagedAttention 的 block 管理源码）
- 进入 `distributed/` → SGLang deep dive（理解 RadixAttention 的 radix tree 实现）
- 进入 `server/` → TensorRT-LLM deep dive（理解 in-flight batching 的调度实现）